# Data Classes from Scratch in Python
## A beginner's tutorial built around a University `Student`

A **data class** is a class whose main job is to **hold information about one thing** — a student, a course, a grade, an address — together with a few useful actions on that information.

In this notebook we build such classes **by hand, step by step**, using only plain Python. We do **not** use decorators or any special library: every line is something you write yourself, so you can see exactly how it works.

### Roadmap

| Part | Topic | New idea |
|---|---|---|
| 1 | Why do we need classes? | storing data in variables, lists and dictionaries — and their problems |
| 2 | Your first class | `class`, objects, attributes |
| 3 | The constructor | `__init__` and `self` |
| 4 | Default values | optional parameters, the "shared list" trap |
| 5 | Methods | functions that belong to an object |
| 6 | Printing objects nicely | `__str__` and `__repr__` |
| 7 | Comparing objects | `==` vs `is`, `__eq__` |
| 8 | Protecting the data | validation, `raise`, `try / except`, getter and setter methods |
| 9 | Computed values | calculating the GPA instead of storing it |
| 10 | Sorting students | `__lt__`, sorting with a key function |
| 11 | Class attributes | data shared by all students, automatic IDs |
| 12 | Objects inside objects | `Course`, `Enrollment`, `Address` (composition) |
| 13 | Inheritance | `Person → Student → GraduateStudent`, `super()` |
| 14 | Saving and loading | `to_dict()`, JSON files |
| 15 | Mini-project | a `StudentRegistry`: add, find, update, delete, reports |
| 16 | Summary | cheat sheet, common mistakes, exercises with solutions |

> **How to use this notebook:** run the cells **from top to bottom** (menu `Kernel → Restart & Run All`). Read the text before each code cell, run the cell, and look at the output. Try changing the values and running it again!

---
## Part 1 — Why do we need classes?

Suppose we want to store information about a student: an **ID**, a **name**, a **major** (field of study), the **year** of study and a **GPA** (grade point average, from 0.0 to 4.0).

### Attempt 1: separate variables

In [1]:
student1_id = "S2026001"
student1_name = "Sara Ahmadi"
student1_major = "Computer Science"
student1_year = 2
student1_gpa = 3.7

student2_id = "S2026002"
student2_name = "Ali Rezaei"
student2_major = "Mathematics"
student2_year = 1
student2_gpa = 3.4

print(student1_name, "studies", student1_major)
print(student2_name, "studies", student2_major)

Sara Ahmadi studies Computer Science
Ali Rezaei studies Mathematics


This works for two students, but imagine **300 students**: 1,500 variables! Nothing connects `student1_name` to `student1_gpa` except our naming habit.

### Attempt 2: a dictionary per student

In [2]:
sara = {"id": "S2026001", "name": "Sara Ahmadi", "major": "Computer Science", "year": 2, "gpa": 3.7}
ali  = {"id": "S2026002", "name": "Ali Rezaei",  "major": "Mathematics",      "year": 1, "gpa": 3.4}

print(sara["name"], "has GPA", sara["gpa"])

# Problem 1: a typo in a key silently creates a NEW key instead of an error
sara["gap"] = 3.9          # we meant "gpa"
print(sara)

# Problem 2: nothing stops impossible values
ali["year"] = -5
print(ali)

# Problem 3: two students can have different keys; nobody checks the "shape"
emma = {"ID": "S2026003", "fullname": "Emma Wilson"}
print(emma)

Sara Ahmadi has GPA 3.7
{'id': 'S2026001', 'name': 'Sara Ahmadi', 'major': 'Computer Science', 'year': 2, 'gpa': 3.7, 'gap': 3.9}
{'id': 'S2026002', 'name': 'Ali Rezaei', 'major': 'Mathematics', 'year': -5, 'gpa': 3.4}
{'ID': 'S2026003', 'fullname': 'Emma Wilson'}


Dictionaries are better, but:
- **typos** create new keys instead of giving an error,
- **nothing checks** that the values make sense (a year of −5?),
- every dictionary can have **different keys**,
- the **actions** (like "add a course" or "compute the GPA") live in separate functions, away from the data.

A **class** solves all of these. It is a **blueprint** that says: *"every student has these pieces of data, these rules, and can do these actions."*

```
            class Student            ← the blueprint (written once)
     ┌───────────────────────────┐
     │ data:    id, name, major, │
     │          year, gpa        │
     │ actions: describe(),      │
     │          add_course() ... │
     └───────────────────────────┘
          │                │
          ▼                ▼
   ┌────────────┐   ┌────────────┐
   │ object #1  │   │ object #2  │   ← objects (instances) made from it
   │ Sara, 3.7  │   │ Ali, 3.4   │
   └────────────┘   └────────────┘
```

---
## Part 2 — Your first class

The smallest possible class. `pass` means "nothing here yet".

In [3]:
class Student:
    pass

s1 = Student()        # create an object (an "instance") from the class
s2 = Student()        # another, separate object

print(type(s1))
print(s1 is s2)       # are they the same object?  -> False

<class '__main__.Student'>
False


We can attach data to an object with a dot. A variable that belongs to an object is called an **attribute**.

In [4]:
s1.name = "Sara Ahmadi"
s1.gpa = 3.7

s2.name = "Ali Rezaei"
s2.gpa = 3.4

print(s1.name, s1.gpa)
print(s2.name, s2.gpa)

Sara Ahmadi 3.7
Ali Rezaei 3.4


Better — but we still add attributes one by one, and nothing guarantees that every student gets a `name` and a `gpa`. If we forget one, we only find out later with an error:

In [5]:
s3 = Student()
s3.name = "Emma Wilson"
# we forgot the gpa ...

try:
    print(s3.gpa)
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: 'Student' object has no attribute 'gpa'


> 💡 **`try` / `except`** — we will use it many times to *show* errors without stopping the notebook. Python runs the `try` block; if an error happens, it jumps to the `except` block that names that kind of error. `as error` gives us the error message.

What we need is a way to say: **"whenever a Student is created, it must receive these values."** That is the job of the constructor.

---
## Part 3 — The constructor: `__init__` and `self`

When you write `Student(...)`, Python automatically calls a special method named **`__init__`** ("initialize"). Its job is to set up the new object's attributes.

```python
class Student:
    def __init__(self, student_id, name, major, year, gpa):
        self.student_id = student_id
        ...
```

**What is `self`?** It is the **new object being created**. Python passes it in automatically as the first argument. `self.name = name` means: *"store the value of the parameter `name` in this object's attribute called `name`."*

```
Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)
           │             │                 │           │   │
           ▼             ▼                 ▼           ▼   ▼
__init__(self, student_id,  name,        major,      year, gpa)
           │
           └── the brand-new, empty object
```

> 📝 Names with two underscores on each side (`__init__`, `__str__`, `__eq__`, …) are called **"dunder" (double underscore) methods**. Python calls them for you in special situations. We'll meet several.

In [6]:
class Student:
    def __init__(self, student_id, name, major, year, gpa):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        self.gpa = gpa

sara = Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)
ali  = Student("S2026002", "Ali Rezaei", "Mathematics", 1, 3.4)

print(sara.name, "-", sara.major, "- year", sara.year, "- GPA", sara.gpa)
print(ali.name, "-", ali.major, "- year", ali.year, "- GPA", ali.gpa)

Sara Ahmadi - Computer Science - year 2 - GPA 3.7
Ali Rezaei - Mathematics - year 1 - GPA 3.4


Now Python **forces** us to give all five values. Forgetting one is an immediate, clear error:

In [7]:
try:
    emma = Student("S2026003", "Emma Wilson", "Physics")    # year and gpa missing
except TypeError as error:
    print("TypeError:", error)

TypeError: Student.__init__() missing 2 required positional arguments: 'year' and 'gpa'


### Keyword arguments
With many parameters it is easy to mix up the order. You can name each argument — then the order doesn't matter and the code reads like a form:

In [8]:
emma = Student(name="Emma Wilson", student_id="S2026003", year=3, gpa=3.2, major="Physics")
print(emma.name, emma.year)

Emma Wilson 3


### Basic operations on an object

| Operation | Code |
|---|---|
| read (access) an attribute | `sara.gpa` |
| change (update) an attribute | `sara.gpa = 3.8` |
| see all attributes | `vars(sara)` |
| read an attribute whose name is in a variable | `getattr(sara, "major")` |

In [9]:
print("Before:", sara.year, sara.gpa)

sara.year = sara.year + 1      # Sara moves up a year
sara.gpa = 3.8                 # and improves her GPA

print("After :", sara.year, sara.gpa)
print("All attributes:", vars(sara))

field_name = "major"
print("getattr:", getattr(sara, field_name))

Before: 2 3.7
After : 3 3.8
All attributes: {'student_id': 'S2026001', 'name': 'Sara Ahmadi', 'major': 'Computer Science', 'year': 3, 'gpa': 3.8}
getattr: Computer Science


Each object has **its own** attributes. Changing Sara does not change Ali:

In [10]:
sara.major = "Software Engineering"
print("Sara:", sara.major)
print("Ali :", ali.major)

Sara: Software Engineering
Ali : Mathematics


---
## Part 4 — Default values

A brand-new student usually starts in **year 1** with a **GPA of 0.0**. We can give parameters **default values**; then the caller may leave them out.

> ⚠️ Rule: parameters **with** defaults must come **after** parameters without defaults.

In [11]:
class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1, gpa=0.0):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        self.gpa = gpa

new_student = Student("S2026010", "Omid Tehrani")
transfer    = Student("S2026011", "Liam Chen", "Physics", year=3, gpa=3.1)

print(vars(new_student))
print(vars(transfer))

{'student_id': 'S2026010', 'name': 'Omid Tehrani', 'major': 'Undeclared', 'year': 1, 'gpa': 0.0}
{'student_id': 'S2026011', 'name': 'Liam Chen', 'major': 'Physics', 'year': 3, 'gpa': 3.1}


### ⚠️ The "shared list" trap

Each student should have a **list of courses**. The obvious idea `courses=[]` has a famous bug. Watch:

In [12]:
class BuggyStudent:
    def __init__(self, name, courses=[]):   # the [] is created only ONCE!
        self.name = name
        self.courses = courses

x = BuggyStudent("Sara")
y = BuggyStudent("Ali")

x.courses.append("CS101")                   # add a course to Sara only...

print("Sara's courses:", x.courses)
print("Ali's courses :", y.courses, "  <-- Ali got Sara's course!")
print("Same list?    :", x.courses is y.courses)

Sara's courses: ['CS101']
Ali's courses : ['CS101']   <-- Ali got Sara's course!
Same list?    : True


**Why?** Python creates the default `[]` **once**, when the `def` line runs — not each time the function is called. So every student who uses the default **shares the same list**.

**The fix:** use `None` as the default, and create a new list *inside* `__init__`:

In [13]:
class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1, courses=None):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        if courses is None:
            courses = []             # a NEW list for every student
        self.courses = courses

x = Student("S2026001", "Sara")
y = Student("S2026002", "Ali")
x.courses.append("CS101")

print("Sara's courses:", x.courses)
print("Ali's courses :", y.courses)
print("Same list?    :", x.courses is y.courses)

Sara's courses: ['CS101']
Ali's courses : []
Same list?    : False


> ✅ **Rule:** never use a list `[]` or dictionary `{}` as a default value. Use `None` and create it inside the function.

---
## Part 5 — Methods: actions that belong to the object

A **method** is a function written inside a class. Its first parameter is always `self`, so it can read and change the object's attributes.

Let's give our student these actions:

| Method | What it does | Kind of operation |
|---|---|---|
| `describe()` | returns a one-line description | access |
| `has_course(code)` | `True` if the student takes that course | search |
| `add_course(code)` | adds a course (no duplicates) | insert |
| `drop_course(code)` | removes a course | delete |
| `change_major(new_major)` | switches major | update |
| `course_count()` | number of courses | access |

In [14]:
class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1, courses=None):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        if courses is None:
            courses = []
        self.courses = courses

    def describe(self):
        return self.name + " (" + self.student_id + "), year " + str(self.year) + ", " + self.major

    def has_course(self, code):
        return code in self.courses

    def add_course(self, code):
        if self.has_course(code):            # a method can call another method through self
            print("  ", self.name, "already takes", code)
            return False
        self.courses.append(code)
        return True

    def drop_course(self, code):
        if not self.has_course(code):
            print("  ", self.name, "does not take", code)
            return False
        self.courses.remove(code)
        return True

    def change_major(self, new_major):
        print("  ", self.name, "changes major from", self.major, "to", new_major)
        self.major = new_major

    def course_count(self):
        return len(self.courses)

In [15]:
sara = Student("S2026001", "Sara Ahmadi", "Computer Science", 2)

print("Access :", sara.describe())

print("Insert :")
sara.add_course("CS101")
sara.add_course("MATH201")
sara.add_course("ENG100")
sara.add_course("CS101")                 # duplicate -> refused
print("         courses =", sara.courses)

print("Search : takes MATH201?", sara.has_course("MATH201"), "| takes ART100?", sara.has_course("ART100"))

print("Delete :")
sara.drop_course("ENG100")
sara.drop_course("ART100")               # not taken -> refused
print("         courses =", sara.courses, "| count =", sara.course_count())

print("Update :")
sara.change_major("Software Engineering")
print("        ", sara.describe())

Access : Sara Ahmadi (S2026001), year 2, Computer Science
Insert :
   Sara Ahmadi already takes CS101
         courses = ['CS101', 'MATH201', 'ENG100']
Search : takes MATH201? True | takes ART100? False
Delete :
   Sara Ahmadi does not take ART100
         courses = ['CS101', 'MATH201'] | count = 2
Update :
   Sara Ahmadi changes major from Computer Science to Software Engineering
         Sara Ahmadi (S2026001), year 2, Software Engineering


**How does `sara.add_course("CS101")` work?** Python translates it to `Student.add_course(sara, "CS101")` — the object before the dot becomes `self`. Let's prove it:

In [16]:
ali = Student("S2026002", "Ali Rezaei", "Mathematics")
Student.add_course(ali, "STAT210")     # the long way
ali.add_course("MATH201")              # the normal way - means exactly the same
print(ali.courses)

['STAT210', 'MATH201']


> 💡 **f-strings.** Building text with `+` and `str()` gets tiring. From now on we use **f-strings**: put `f` before the quotes and write variables inside `{ }`.
> `f"{self.name} has GPA {self.gpa}"` is the same as `self.name + " has GPA " + str(self.gpa)`.

In [17]:
name, gpa = "Sara", 3.756
print(f"{name} has GPA {gpa}")
print(f"{name} has GPA {gpa:.2f}")      # :.2f -> show 2 digits after the decimal point

Sara has GPA 3.756
Sara has GPA 3.76


---
## Part 6 — Printing objects nicely: `__str__` and `__repr__`

What happens if we just `print` a student?

In [18]:
print(sara)

Python doesn't know how to show a student, so it prints the class name and a **memory address**. We teach it with two dunder methods:

| Method | Used by | Audience | Style |
|---|---|---|---|
| `__str__` | `print(obj)`, `str(obj)`, f-strings | end users | friendly text |
| `__repr__` | the notebook when you just type `obj`, lists of objects, `repr(obj)` | programmers | precise, looks like the code to create it |

Both must **return** a string (not print it).

In [19]:
class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1, gpa=0.0):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        self.gpa = gpa

    def __str__(self):
        return f"{self.name} ({self.student_id}) - {self.major}, year {self.year}, GPA {self.gpa}"

    def __repr__(self):
        return (f"Student('{self.student_id}', '{self.name}', '{self.major}', "
                f"{self.year}, {self.gpa})")

sara = Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)
ali  = Student("S2026002", "Ali Rezaei", "Mathematics", 1, 3.4)

print(sara)                    # uses __str__
print(f"Top student: {sara}")  # f-strings also use __str__
print(repr(sara))              # uses __repr__
print([sara, ali])             # a list shows its items with __repr__

Sara Ahmadi (S2026001) - Computer Science, year 2, GPA 3.7
Top student: Sara Ahmadi (S2026001) - Computer Science, year 2, GPA 3.7
Student('S2026001', 'Sara Ahmadi', 'Computer Science', 2, 3.7)
[Student('S2026001', 'Sara Ahmadi', 'Computer Science', 2, 3.7), Student('S2026002', 'Ali Rezaei', 'Mathematics', 1, 3.4)]


In a notebook, if the **last line** of a cell is just an object, Jupyter displays it with `__repr__`:

In [20]:
ali

Student('S2026002', 'Ali Rezaei', 'Mathematics', 1, 3.4)

> 💡 If you write only **one** of the two, write `__repr__`: Python uses it for `print` too when `__str__` is missing.

---
## Part 7 — Comparing objects: `==`, `is`, and `__eq__`

The university database loaded Sara's record **twice**. Are the two records equal?

In [21]:
record_a = Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)
record_b = Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)

print("record_a == record_b :", record_a == record_b)
print("record_a is record_b :", record_a is record_b)

record_a == record_b : False
record_a is record_b : False


Both are `False`! There are two different questions we can ask about objects:

| Operator | Question | Default for our classes |
|---|---|---|
| `a is b` | are `a` and `b` **the very same object** in memory? | identity |
| `a == b` | do `a` and `b` contain **equal data**? | **also identity**, unless we define `__eq__` |

```
record_a ──►  [ S2026001 | Sara | ... ]      two separate objects,
record_b ──►  [ S2026001 | Sara | ... ]      same contents

record_c = record_a
record_c ──┘  (record_c points to the SAME object as record_a)
```

We define **`__eq__(self, other)`**: Python calls it for `a == b` (with `self = a`, `other = b`). It must return `True` or `False`.

**Design decision:** when are two students "equal"? A university would say: *when they have the same student ID*. The ID is unique; names can repeat (two students can both be called "Ali Rezaei").

In [22]:
class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1, gpa=0.0):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        self.gpa = gpa

    def __str__(self):
        return f"{self.name} ({self.student_id}) - {self.major}, year {self.year}, GPA {self.gpa}"

    def __repr__(self):
        return f"Student('{self.student_id}', '{self.name}', '{self.major}', {self.year}, {self.gpa})"

    def __eq__(self, other):
        if not isinstance(other, Student):    # comparing with a non-student -> not equal
            return False
        return self.student_id == other.student_id

record_a = Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)
record_b = Student("S2026001", "Sara Ahmadi", "Computer Science", 2, 3.7)
record_c = record_a
other_ali_1 = Student("S2026002", "Ali Rezaei", "Mathematics")
other_ali_2 = Student("S2026099", "Ali Rezaei", "History")

print("a == b (same ID)            :", record_a == record_b)
print("a is b (same object?)       :", record_a is record_b)
print("a is c (same object?)       :", record_a is record_c)
print("two Alis, different IDs     :", other_ali_1 == other_ali_2)
print("student == 'S2026001'       :", record_a == "S2026001")
print("a != ali (Python uses __eq__):", record_a != other_ali_1)

a == b (same ID)            : True
a is b (same object?)       : False
a is c (same object?)       : True
two Alis, different IDs     : False
student == 'S2026001'       : False
a != ali (Python uses __eq__): True


### Equality makes searching in lists work

`in`, `list.count()`, `list.index()` and `list.remove()` all use `==`, so they now work with students:

In [23]:
roster = [record_a, other_ali_1]
lookup = Student("S2026001", "?")           # only the ID matters for equality

print("Is S2026001 in the roster?", lookup in roster)
print("Position in the roster   :", roster.index(lookup))
roster.remove(lookup)                       # removes Sara
print("Roster after remove      :", roster)

Is S2026001 in the roster? True
Position in the roster   : 0
Roster after remove      : [Student('S2026002', 'Ali Rezaei', 'Mathematics', 1, 0.0)]


### Sets and dictionary keys: `__hash__`

To put objects in a **set** or use them as **dictionary keys**, Python needs a *hash*: a number computed from the object. **Important rule:** objects that are equal must have the same hash. When you write `__eq__`, Python turns hashing off for safety, so this fails:

In [24]:
try:
    unique = {record_a, record_b}
except TypeError as error:
    print("TypeError:", error)

TypeError: unhashable type: 'Student'


Since our equality is based only on `student_id`, we hash the `student_id`. We add one small method to the class:

In [25]:
def student_hash(self):
    return hash(self.student_id)

Student.__hash__ = student_hash     # attach the method to the existing class
                                    # (normally you would just write it inside the class)

unique = {record_a, record_b, other_ali_1}
print("Unique students:", len(unique))

gpa_lookup = {record_a: 3.7, other_ali_1: 3.4}
print("GPA of S2026001:", gpa_lookup[Student("S2026001", "?")])

Unique students: 2
GPA of S2026001: 3.7


> ⚠️ Only hash attributes that **never change**. A student ID never changes; a GPA does. If the hash changed while the object was inside a set, the set could no longer find it.

---
## Part 8 — Protecting the data: validation

Right now nothing stops nonsense:

In [26]:
nonsense = Student("hello", "", "Physics", -3, 17.5)
print(nonsense)

 (hello) - Physics, year -3, GPA 17.5


A good data class **checks its data** and refuses bad values. To refuse, we **raise an error** with `raise ValueError("message")`. That stops the creation of the object and shows the message.

University rules:

| Attribute | Rule |
|---|---|
| `student_id` | the letter `S` followed by exactly 7 digits, e.g. `S2026001` |
| `name` | not empty; we also clean extra spaces and capitalize it |
| `year` | a whole number from 1 to 6 |
| `gpa` | a number from 0.0 to 4.0 |

We put each check in its own small helper method, and `__init__` calls them. Helper methods that are meant only for use *inside* the class start with an underscore `_` — a **convention** meaning "internal, please don't call this from outside".

In [27]:
class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1, gpa=0.0):
        self.student_id = self._check_id(student_id)
        self.name = self._clean_name(name)
        self.major = major
        self.year = self._check_year(year)
        self._gpa = self._check_gpa(gpa)       # underscore: please use get_gpa / set_gpa

    # ---------------- validation helpers ----------------
    def _check_id(self, student_id):
        ok = (isinstance(student_id, str) and len(student_id) == 8
              and student_id[0] == "S" and student_id[1:].isdigit())
        if not ok:
            raise ValueError(f"invalid student ID {student_id!r}: expected 'S' + 7 digits")
        return student_id

    def _clean_name(self, name):
        cleaned = " ".join(name.split())      # remove extra spaces
        if cleaned == "":
            raise ValueError("name must not be empty")
        return cleaned.title()                # 'sara AHMADI' -> 'Sara Ahmadi'

    def _check_year(self, year):
        if not isinstance(year, int) or year < 1 or year > 6:
            raise ValueError(f"year must be a whole number from 1 to 6, got {year!r}")
        return year

    def _check_gpa(self, gpa):
        if not isinstance(gpa, (int, float)) or gpa < 0.0 or gpa > 4.0:
            raise ValueError(f"GPA must be between 0.0 and 4.0, got {gpa!r}")
        return round(float(gpa), 2)

    # ---------------- getter and setter for the GPA ----------------
    def get_gpa(self):
        return self._gpa

    def set_gpa(self, new_gpa):
        self._gpa = self._check_gpa(new_gpa)  # every change is checked too

    # ---------------- other updates also go through checks ----------------
    def promote(self):
        self.year = self._check_year(self.year + 1)

    # ---------------- printing / comparing ----------------
    def __str__(self):
        return f"{self.name} ({self.student_id}) - {self.major}, year {self.year}, GPA {self._gpa}"

    def __repr__(self):
        return f"Student('{self.student_id}', '{self.name}', '{self.major}', {self.year}, {self._gpa})"

    def __eq__(self, other):
        return isinstance(other, Student) and self.student_id == other.student_id

    def __hash__(self):
        return hash(self.student_id)

> 💡 `{student_id!r}` inside an f-string shows the value **with quotes** if it is text (`'2026001'`), which makes error messages clearer.

In [28]:
sara = Student("S2026001", "   sara    AHMADI  ", "Computer Science", 2, 3.666)
print("Cleaned and rounded:", sara)

bad_inputs = [
    ("2026001",  "Ali",  "Math",    1, 3.0),    # missing the 'S'
    ("S2026002", "   ",  "Math",    1, 3.0),    # empty name
    ("S2026003", "Emma", "Physics", 9, 3.0),    # year too big
    ("S2026004", "Liam", "Biology", 2, 4.5),    # GPA too big
    ("S2026005", "Neda", "Biology", 2.5, 3.0),  # year not a whole number
]
for data in bad_inputs:
    try:
        Student(*data)            # *data unpacks the tuple into separate arguments
    except ValueError as error:
        print("Refused:", error)

Cleaned and rounded: Sara Ahmadi (S2026001) - Computer Science, year 2, GPA 3.67
Refused: invalid student ID '2026001': expected 'S' + 7 digits
Refused: name must not be empty
Refused: year must be a whole number from 1 to 6, got 9
Refused: GPA must be between 0.0 and 4.0, got 4.5
Refused: year must be a whole number from 1 to 6, got 2.5


### Updating through methods

We read and change the GPA through **`get_gpa()`** and **`set_gpa()`** so every change is checked:

In [29]:
print("GPA now:", sara.get_gpa())
sara.set_gpa(3.9)
print("GPA after set_gpa(3.9):", sara.get_gpa())

try:
    sara.set_gpa(5.0)
except ValueError as error:
    print("Refused:", error, "| GPA is still", sara.get_gpa())

for _ in range(5):            # promote Sara several times
    try:
        sara.promote()
        print("Promoted to year", sara.year)
    except ValueError as error:
        print("Cannot promote:", error)
        break

GPA now: 3.67
GPA after set_gpa(3.9): 3.9
Refused: GPA must be between 0.0 and 4.0, got 5.0 | GPA is still 3.9
Promoted to year 3
Promoted to year 4
Promoted to year 5
Promoted to year 6
Cannot promote: year must be a whole number from 1 to 6, got 7


> ⚠️ Python **does not truly prevent** anyone from writing `sara._gpa = 99`. The underscore is a polite "*keep out*" sign for other programmers, not a lock. In Python we trust programmers to respect the convention.

---
## Part 9 — Computed values: calculate the GPA, don't store it

In a real university the GPA is not typed in — it is **computed from the grades**. If we stored the GPA *and* the grades separately, they could disagree (someone updates a grade but forgets the GPA). The safe design:

> **Store** what you know (the grades). **Compute** what follows from it (GPA, credits, academic standing) with a method, every time it is asked for.

### How a GPA is computed
Each letter grade is worth some **points**, and each course is worth some **credits** (hours per week). The GPA is the credit-weighted average:

$$\text{GPA} = \frac{\sum (\text{points} \times \text{credits})}{\sum \text{credits}}$$

| Letter | A | A- | B+ | B | B- | C+ | C | D | F |
|---|---|---|---|---|---|---|---|---|---|
| Points | 4.0 | 3.7 | 3.3 | 3.0 | 2.7 | 2.3 | 2.0 | 1.0 | 0.0 |

**Example:** CS101 (4 credits, A = 4.0), MATH201 (3 credits, B+ = 3.3)

$$\text{GPA} = \frac{4.0 \times 4 + 3.3 \times 3}{4 + 3} = \frac{25.9}{7} = 3.70$$

We store grades in a **dictionary**: `{ course_code : [credits, letter] }`.

In [30]:
GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7,
                "C+": 2.3, "C": 2.0, "D": 1.0, "F": 0.0}

class Student:
    def __init__(self, student_id, name, major="Undeclared", year=1):
        self.student_id = student_id
        self.name = name
        self.major = major
        self.year = year
        self.grades = {}                     # course code -> [credits, letter]

    # ----- insert / update / delete grades -----
    def add_grade(self, course_code, credits, letter):
        if letter not in GRADE_POINTS:
            raise ValueError(f"unknown letter grade {letter!r}")
        if credits < 1 or credits > 6:
            raise ValueError("credits must be from 1 to 6")
        self.grades[course_code] = [credits, letter]   # new course -> insert, existing -> update

    def remove_grade(self, course_code):
        if course_code in self.grades:
            del self.grades[course_code]
            return True
        return False

    # ----- access / search -----
    def get_grade(self, course_code):
        if course_code in self.grades:
            return self.grades[course_code][1]
        return None

    # ----- computed values -----
    def total_credits(self):
        total = 0
        for course_code in self.grades:
            credits = self.grades[course_code][0]
            total = total + credits
        return total

    def credits_earned(self):            # an F does not earn credits
        total = 0
        for course_code in self.grades:
            credits, letter = self.grades[course_code]
            if letter != "F":
                total = total + credits
        return total

    def gpa(self):
        if self.total_credits() == 0:
            return 0.0
        weighted_points = 0
        for course_code in self.grades:
            credits, letter = self.grades[course_code]
            weighted_points = weighted_points + GRADE_POINTS[letter] * credits
        return round(weighted_points / self.total_credits(), 2)

    def standing(self):
        if not self.grades:
            return "No grades yet"
        if self.gpa() >= 3.5:
            return "Dean's List"
        if self.gpa() >= 2.0:
            return "Good standing"
        return "Academic probation"

    def transcript(self):
        lines = [f"Transcript: {self.name} ({self.student_id}), {self.major}, year {self.year}"]
        for course_code in sorted(self.grades):
            credits, letter = self.grades[course_code]
            lines.append(f"   {course_code:<8} {credits} credits   {letter:<2}  ({GRADE_POINTS[letter]})")
        lines.append(f"   GPA {self.gpa():.2f} | credits earned {self.credits_earned()} | {self.standing()}")
        return "\n".join(lines)

    def __str__(self):
        return f"{self.name} ({self.student_id}), GPA {self.gpa()}"

In [31]:
sara = Student("S2026001", "Sara Ahmadi", "Computer Science", 2)
print("Before any grades:", sara, "|", sara.standing())

sara.add_grade("CS101", 4, "A")
sara.add_grade("MATH201", 3, "B+")
print("\n" + sara.transcript())       # matches the hand calculation: 3.70

Before any grades: Sara Ahmadi (S2026001), GPA 0.0 | No grades yet

Transcript: Sara Ahmadi (S2026001), Computer Science, year 2
   CS101    4 credits   A   (4.0)
   MATH201  3 credits   B+  (3.3)
   GPA 3.70 | credits earned 7 | Dean's List


In [32]:
print("Insert: a failing grade in HIST100")
sara.add_grade("HIST100", 2, "F")
print(sara.transcript())

print("\nUpdate: Sara retakes MATH201 and gets an A")
sara.add_grade("MATH201", 3, "A")
print("Search: grade in MATH201 =", sara.get_grade("MATH201"), "| grade in ART100 =", sara.get_grade("ART100"))

print("\nDelete: HIST100 is removed after an appeal")
sara.remove_grade("HIST100")
print(sara.transcript())

Insert: a failing grade in HIST100
Transcript: Sara Ahmadi (S2026001), Computer Science, year 2
   CS101    4 credits   A   (4.0)
   HIST100  2 credits   F   (0.0)
   MATH201  3 credits   B+  (3.3)
   GPA 2.88 | credits earned 7 | Good standing

Update: Sara retakes MATH201 and gets an A
Search: grade in MATH201 = A | grade in ART100 = None

Delete: HIST100 is removed after an appeal
Transcript: Sara Ahmadi (S2026001), Computer Science, year 2
   CS101    4 credits   A   (4.0)
   MATH201  3 credits   A   (4.0)
   GPA 4.00 | credits earned 7 | Dean's List


Notice: we **never** set the GPA ourselves, yet it is always correct, because `gpa()` recomputes it from the grades every time.

---
## Part 10 — Sorting students

Let's make a small class of students and try to sort them:

In [33]:
def make_student(student_id, name, major, year, grade_list):
    """Helper: create a student and add a list of (course, credits, letter) grades."""
    s = Student(student_id, name, major, year)
    for course_code, credits, letter in grade_list:
        s.add_grade(course_code, credits, letter)
    return s

students = [
    make_student("S2026003", "Emma Wilson", "Physics", 3,
                 [("PHYS110", 4, "B"), ("MATH201", 3, "C+")]),
    make_student("S2026001", "Sara Ahmadi", "Computer Science", 2,
                 [("CS101", 4, "A"), ("MATH201", 3, "B+")]),
    make_student("S2026002", "Ali Rezaei", "Mathematics", 1,
                 [("MATH201", 3, "A"), ("STAT210", 3, "A-")]),
]

try:
    sorted(students)
except TypeError as error:
    print("TypeError:", error)

TypeError: '<' not supported between instances of 'Student' and 'Student'


Python doesn't know what "smaller student" means. There are two ways to tell it.

### Way 1: a key function (most common)
`sorted(list, key=some_function)` calls the function on **each** item and sorts by the values it returns:

In [34]:
def by_gpa(student):
    return student.gpa()

def by_name(student):
    return student.name

print("By name:")
for s in sorted(students, key=by_name):
    print("  ", s)

print("By GPA, highest first:")
for s in sorted(students, key=by_gpa, reverse=True):
    print("  ", s)

best = max(students, key=by_gpa)        # max and min accept a key too
print("Best student:", best.name)

By name:
   Ali Rezaei (S2026002), GPA 3.85
   Emma Wilson (S2026003), GPA 2.7
   Sara Ahmadi (S2026001), GPA 3.7
By GPA, highest first:
   Ali Rezaei (S2026002), GPA 3.85
   Sara Ahmadi (S2026001), GPA 3.7
   Emma Wilson (S2026003), GPA 2.7
Best student: Ali Rezaei


### Way 2: give the class a natural order with `__lt__`

`a < b` calls `a.__lt__(b)` ("less than"). `sorted()` only needs `<`. Let's say the *natural* order of students is **by student ID** (the order they were admitted). We add the method to our class:

In [35]:
def student_less_than(self, other):
    return self.student_id < other.student_id

Student.__lt__ = student_less_than      # attach (normally written inside the class)

print("Natural order (by ID):")
for s in sorted(students):
    print("  ", s.student_id, s.name)

print("\nsara < ali ?", students[1] < students[2])

Natural order (by ID):
   S2026001 Sara Ahmadi
   S2026002 Ali Rezaei
   S2026003 Emma Wilson

sara < ali ? True


> 💡 Use `__lt__` for the **one obvious** order of your objects; use `key=` for any other order you need. Both can be used in the same program.

---
## Part 11 — Class attributes: data shared by all students

So far every attribute belonged to **one** object (`self.name`). These are **instance attributes**.

Some data belongs to the **class as a whole**:
- the university's name — the same for every student;
- the maximum number of years — a rule for everyone;
- a counter of how many students have been admitted, used to give each new student the **next ID automatically**.

Variables written directly in the class body (not inside a method) are **class attributes**. They are shared by all objects.

```
            Student (class)
   ┌───────────────────────────────────┐
   │ UNIVERSITY = "Example University" │  ← class attributes (one copy)
   │ next_number = 3                   │
   └───────────────────────────────────┘
        ▲              ▲
   ┌────┴─────┐   ┌────┴─────┐
   │ name     │   │ name     │           ← instance attributes (one copy per object)
   │ id       │   │ id       │
   └──────────┘   └──────────┘
```

In [36]:
class Student:
    UNIVERSITY = "Example University"     # class attribute: a constant
    MAX_YEAR = 6                          # class attribute: a rule
    next_number = 1                       # class attribute: a shared counter

    def __init__(self, name, major="Undeclared"):
        # build an ID like S2026001 from the shared counter
        self.student_id = "S2026" + str(Student.next_number).zfill(3)
        Student.next_number = Student.next_number + 1      # update the SHARED counter
        self.name = name
        self.major = major
        self.year = 1

    def email(self):
        return self.student_id.lower() + "@uni.example.edu"

    def __str__(self):
        return f"{self.student_id} {self.name} <{self.email()}> at {Student.UNIVERSITY}"

new_students = [Student("Sara Ahmadi", "Computer Science"),
                Student("Ali Rezaei", "Mathematics"),
                Student("Emma Wilson", "Physics")]
for s in new_students:
    print(s)

print("\nNext ID number to give:", Student.next_number)
print("Read through an object too:", new_students[0].MAX_YEAR)

S2026001 Sara Ahmadi <s2026001@uni.example.edu> at Example University
S2026002 Ali Rezaei <s2026002@uni.example.edu> at Example University
S2026003 Emma Wilson <s2026003@uni.example.edu> at Example University

Next ID number to give: 4
Read through an object too: 6


> ⚠️ **Always change class attributes through the class name** (`Student.next_number = ...`). Writing `self.next_number = ...` would create a *new instance attribute* on that one object and leave the shared counter unchanged:

In [37]:
s = new_students[0]
s.next_number = 999                  # creates a private copy on Sara only!
print("Sara's own attribute :", s.next_number)
print("The shared counter   :", Student.next_number)

Sara's own attribute : 999
The shared counter   : 4


---
## Part 12 — Objects inside objects (composition)

A real student record is made of **smaller records**:

```
Student ──has one──► Address
   │
   └──has many──► Enrollment ──refers to──► Course
                   (semester,              (code, title,
                    grade)                  credits, department)
```

Each box becomes its own small class. A `Student` **contains** an `Address` object and a **list** of `Enrollment` objects; each `Enrollment` **contains** a `Course`. This is called **composition** — building bigger objects out of smaller ones. Each class stays small and easy to understand.

This is the **final model** we will use for the rest of the notebook.

In [38]:
class Course:
    def __init__(self, code, title, credits, department):
        if credits < 1 or credits > 6:
            raise ValueError(f"{code}: credits must be from 1 to 6")
        self.code = code
        self.title = title
        self.credits = credits
        self.department = department

    def __str__(self):
        return f"{self.code} {self.title} ({self.credits} cr)"

    def __repr__(self):
        return f"Course('{self.code}', '{self.title}', {self.credits}, '{self.department}')"

    def __eq__(self, other):
        return isinstance(other, Course) and self.code == other.code

    def __hash__(self):
        return hash(self.code)


class Address:
    def __init__(self, street, city, postal_code, country="Iran"):
        self.street = street
        self.city = city
        self.postal_code = postal_code
        self.country = country

    def __str__(self):
        return f"{self.street}, {self.city} {self.postal_code}, {self.country}"


class Enrollment:
    def __init__(self, course, semester, grade=None):
        self.course = course          # a Course object
        self.semester = semester      # e.g. "2025-Fall"
        self.grade = grade            # None means "still in progress"

    def is_completed(self):
        return self.grade is not None

    def __str__(self):
        shown = self.grade if self.grade is not None else "in progress"
        return f"{self.semester}: {self.course} -> {shown}"

In [39]:
class Student:
    def __init__(self, student_id, first_name, last_name, major, year=1, address=None):
        if len(student_id) != 8 or student_id[0] != "S" or not student_id[1:].isdigit():
            raise ValueError(f"invalid student ID {student_id!r}")
        if year < 1 or year > 6:
            raise ValueError("year must be from 1 to 6")
        self.student_id = student_id
        self.first_name = first_name
        self.last_name = last_name
        self.major = major
        self.year = year
        self.address = address        # an Address object, or None
        self.enrollments = []         # a list of Enrollment objects

    # ---------- computed ----------
    def full_name(self):
        return self.first_name + " " + self.last_name

    def email(self):
        return self.student_id.lower() + "@uni.example.edu"

    def gpa(self):
        total_credits = 0
        weighted_points = 0
        for e in self.enrollments:
            if e.is_completed():
                total_credits = total_credits + e.course.credits
                weighted_points = weighted_points + GRADE_POINTS[e.grade] * e.course.credits
        if total_credits == 0:
            return 0.0
        return round(weighted_points / total_credits, 2)

    def credits_earned(self):
        total = 0
        for e in self.enrollments:
            if e.is_completed() and e.grade != "F":
                total = total + e.course.credits
        return total

    # ---------- search ----------
    def find_enrollment(self, course_code, semester=None):
        for e in self.enrollments:
            if e.course.code == course_code:
                if semester is None or e.semester == semester:
                    return e
        return None

    # ---------- insert ----------
    def enroll(self, course, semester):
        if self.find_enrollment(course.code, semester) is not None:
            raise ValueError(f"{self.full_name()} is already enrolled in {course.code} for {semester}")
        new_enrollment = Enrollment(course, semester)
        self.enrollments.append(new_enrollment)
        return new_enrollment

    # ---------- update ----------
    def set_grade(self, course_code, semester, grade):
        if grade not in GRADE_POINTS:
            raise ValueError(f"unknown grade {grade!r}")
        e = self.find_enrollment(course_code, semester)
        if e is None:
            raise LookupError(f"{self.full_name()} is not enrolled in {course_code} ({semester})")
        e.grade = grade

    # ---------- delete ----------
    def drop(self, course_code, semester):
        e = self.find_enrollment(course_code, semester)
        if e is None or e.is_completed():      # cannot drop a finished course
            return False
        self.enrollments.remove(e)
        return True

    # ---------- printing / comparing ----------
    def transcript(self):
        lines = [f"{self.full_name()} ({self.student_id}) - {self.major}, year {self.year}"]
        if self.address is not None:
            lines.append(f"   address: {self.address}")
        for e in self.enrollments:
            lines.append(f"   {e}")
        lines.append(f"   GPA {self.gpa():.2f} | credits earned {self.credits_earned()}")
        return "\n".join(lines)

    def __str__(self):
        return f"{self.full_name()} ({self.student_id}), {self.major}, GPA {self.gpa()}"

    def __repr__(self):
        return f"Student('{self.student_id}', '{self.first_name}', '{self.last_name}', '{self.major}', {self.year})"

    def __eq__(self, other):
        return isinstance(other, Student) and self.student_id == other.student_id

    def __hash__(self):
        return hash(self.student_id)

print("Model ready: Course, Address, Enrollment, Student")

Model ready: Course, Address, Enrollment, Student


### The course catalogue
We keep all courses in a dictionary, so we can find a course by its code quickly: `CATALOGUE["CS101"]`.

In [40]:
CATALOGUE = {}
for course in [
    Course("CS101",   "Introduction to Programming", 4, "Computer Science"),
    Course("CS201",   "Data Structures",             4, "Computer Science"),
    Course("MATH201", "Linear Algebra",              3, "Mathematics"),
    Course("STAT210", "Probability and Statistics",  3, "Mathematics"),
    Course("PHYS110", "Physics I",                   4, "Physics"),
    Course("ENG100",  "Academic Writing",            2, "Humanities"),
]:
    CATALOGUE[course.code] = course

for code in CATALOGUE:
    print(CATALOGUE[code])

CS101 Introduction to Programming (4 cr)
CS201 Data Structures (4 cr)
MATH201 Linear Algebra (3 cr)
STAT210 Probability and Statistics (3 cr)
PHYS110 Physics I (4 cr)
ENG100 Academic Writing (2 cr)


In [41]:
sara = Student("S2026001", "Sara", "Ahmadi", "Computer Science", 2,
               Address("12 Valiasr St", "Tehran", "1599616313"))

print("Insert: enroll in three courses for 2025-Fall and one for 2026-Fall")
for code in ["CS101", "MATH201", "ENG100"]:
    sara.enroll(CATALOGUE[code], "2025-Fall")
sara.enroll(CATALOGUE["CS201"], "2026-Fall")

print("Update: grades arrive for 2025-Fall")
sara.set_grade("CS101", "2025-Fall", "A")
sara.set_grade("MATH201", "2025-Fall", "B+")
sara.set_grade("ENG100", "2025-Fall", "A-")

print()
print(sara.transcript())

Insert: enroll in three courses for 2025-Fall and one for 2026-Fall
Update: grades arrive for 2025-Fall

Sara Ahmadi (S2026001) - Computer Science, year 2
   address: 12 Valiasr St, Tehran 1599616313, Iran
   2025-Fall: CS101 Introduction to Programming (4 cr) -> A
   2025-Fall: MATH201 Linear Algebra (3 cr) -> B+
   2025-Fall: ENG100 Academic Writing (2 cr) -> A-
   2026-Fall: CS201 Data Structures (4 cr) -> in progress
   GPA 3.70 | credits earned 9


In [42]:
print("Access through the chain of objects:")
print("  city             :", sara.address.city)
print("  first course     :", sara.enrollments[0].course.title)
print("  its credits      :", sara.enrollments[0].course.credits)

print("Search:")
print("  MATH201          :", sara.find_enrollment("MATH201"))
print("  PHYS110          :", sara.find_enrollment("PHYS110"))

print("Delete:")
print("  drop CS201 (in progress):", sara.drop("CS201", "2026-Fall"))
print("  drop CS101 (finished)   :", sara.drop("CS101", "2025-Fall"))

print("Errors:")
try:
    sara.enroll(CATALOGUE["CS101"], "2025-Fall")
except ValueError as error:
    print("  ValueError:", error)
try:
    sara.set_grade("PHYS110", "2025-Fall", "A")
except LookupError as error:
    print("  LookupError:", error)

Access through the chain of objects:
  city             : Tehran
  first course     : Introduction to Programming
  its credits      : 4
Search:
  MATH201          : 2025-Fall: MATH201 Linear Algebra (3 cr) -> B+
  PHYS110          : None
Delete:
  drop CS201 (in progress): True
  drop CS101 (finished)   : False
Errors:
  ValueError: Sara Ahmadi is already enrolled in CS101 for 2025-Fall
  LookupError: Sara Ahmadi is not enrolled in PHYS110 (2025-Fall)


### Shared objects
There is only **one** `CS101` course object in the catalogue; every student who enrolls refers to that same object. If the university corrects the course title, every transcript sees the change:

In [43]:
ali = Student("S2026002", "Ali", "Rezaei", "Mathematics", 1)
ali.enroll(CATALOGUE["CS101"], "2025-Fall")

print("Same course object?", sara.enrollments[0].course is ali.enrollments[0].course)
CATALOGUE["CS101"].title = "Intro to Programming with Python"
print("Sara sees:", sara.enrollments[0].course.title)
print("Ali sees :", ali.enrollments[0].course.title)

Same course object? True
Sara sees: Intro to Programming with Python
Ali sees : Intro to Programming with Python


---
## Part 13 — Inheritance: `Person → Student → GraduateStudent`

Students and professors have things in common: a name and an email. Instead of copying that code into each class, we write it **once** in a **parent class** `Person`, and let other classes **inherit** from it.

```
                 Person
            (name, email, describe())
              /                 \
        Student                 Professor
   (+ student_id, major,      (+ staff_id, department)
      year, describe())
           |
    GraduateStudent
   (+ advisor, thesis_title, describe())
```

- `class Student(Person):` means "a Student **is a** Person" — it gets all of `Person`'s methods.
- **`super().__init__(...)`** calls the parent's `__init__`, so the parent's attributes are set up without repeating the code.
- A child can **override** (replace) a method by defining one with the same name. It can still call the parent's version with `super().method()`.

In [44]:
class Person:
    def __init__(self, name, email):
        self.name = name
        self.email = email

    def describe(self):
        return f"{self.name} <{self.email}>"

    def __str__(self):
        return self.describe()


class Undergraduate(Person):
    def __init__(self, name, email, student_id, major, year=1):
        super().__init__(name, email)       # let Person set name and email
        self.student_id = student_id        # then add Student-only attributes
        self.major = major
        self.year = year

    def describe(self):                     # override, but reuse the parent's text
        return super().describe() + f", {self.major} student, year {self.year}"


class GraduateStudent(Undergraduate):
    def __init__(self, name, email, student_id, major, year, advisor, thesis_title="TBD"):
        super().__init__(name, email, student_id, major, year)
        self.advisor = advisor
        self.thesis_title = thesis_title

    def describe(self):
        return super().describe() + f", advisor {self.advisor}, thesis '{self.thesis_title}'"


class Professor(Person):
    def __init__(self, name, email, staff_id, department):
        super().__init__(name, email)
        self.staff_id = staff_id
        self.department = department

    def describe(self):
        return "Prof. " + super().describe() + f", {self.department} department"


people = [
    Undergraduate("Sara Ahmadi", "s2026001@uni.example.edu", "S2026001", "Computer Science", 2),
    GraduateStudent("Neda Karimi", "s2024107@uni.example.edu", "S2024107", "Physics", 5,
                    "Prof. Chen", "Quantum Dots"),
    Professor("Liam Chen", "l.chen@uni.example.edu", "P0042", "Physics"),
]

for p in people:
    print(type(p).__name__, "->", p)       # print uses __str__, which uses each class's describe()

Undergraduate -> Sara Ahmadi <s2026001@uni.example.edu>, Computer Science student, year 2
GraduateStudent -> Neda Karimi <s2024107@uni.example.edu>, Physics student, year 5, advisor Prof. Chen, thesis 'Quantum Dots'
Professor -> Prof. Liam Chen <l.chen@uni.example.edu>, Physics department


The loop called the **same method name** `describe()` on different kinds of objects, and each object used **its own version**. This is called **polymorphism** ("many forms").

`isinstance(obj, SomeClass)` asks "is this object a `SomeClass` (or a child of it)?":

In [45]:
neda = people[1]
print("Neda is a GraduateStudent?", isinstance(neda, GraduateStudent))
print("Neda is an Undergraduate (parent class)?", isinstance(neda, Undergraduate))
print("Neda is a Person?", isinstance(neda, Person))
print("Neda is a Professor?", isinstance(neda, Professor))

print("\nNeda has attributes from all levels:")
print("  from Person         :", neda.name, neda.email)
print("  from Undergraduate  :", neda.student_id, neda.major, neda.year)
print("  from GraduateStudent:", neda.advisor, neda.thesis_title)

Neda is a GraduateStudent? True
Neda is an Undergraduate (parent class)? True
Neda is a Person? True
Neda is a Professor? False

Neda has attributes from all levels:
  from Person         : Neda Karimi s2024107@uni.example.edu
  from Undergraduate  : S2024107 Physics 5
  from GraduateStudent: Prof. Chen Quantum Dots


> ⚠️ If a child class has its own `__init__` and **forgets `super().__init__(...)`**, the parent's attributes are never created:

In [46]:
class ForgetfulStudent(Person):
    def __init__(self, name, email, student_id):
        self.student_id = student_id        # forgot super().__init__(name, email)!

f = ForgetfulStudent("Omid", "omid@uni.example.edu", "S2026006")
try:
    print(f.describe())
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: 'ForgetfulStudent' object has no attribute 'name'


---
## Part 14 — Saving and loading: dictionaries and JSON

Objects disappear when the program stops. To **save** them we convert them to a simple format that can be written to a file. The most common format is **JSON**, which looks almost exactly like Python dictionaries and lists.

Plan:
1. give each class a **`to_dict()`** method that turns the object into a plain dictionary;
2. write plain **functions** that rebuild objects from those dictionaries;
3. use Python's built-in **`json`** module to write and read a file.

```
  Student object ──to_dict()──► dict ──json.dump──►  students.json  (file)
  Student object ◄─from_dict── dict ◄──json.load───  students.json
```

In [47]:
def address_to_dict(address):
    if address is None:
        return None
    return {"street": address.street, "city": address.city,
            "postal_code": address.postal_code, "country": address.country}

def student_to_dict(student):
    enrollment_list = []
    for e in student.enrollments:
        enrollment_list.append({"course_code": e.course.code,   # save only the code;
                                "semester": e.semester,          # the course itself lives
                                "grade": e.grade})               # in the catalogue
    return {
        "student_id": student.student_id,
        "first_name": student.first_name,
        "last_name": student.last_name,
        "major": student.major,
        "year": student.year,
        "address": address_to_dict(student.address),
        "enrollments": enrollment_list,
    }

# Attach as a method so we can write student.to_dict()
Student.to_dict = student_to_dict

d = sara.to_dict()
for key in d:
    print(f"{key:<12}: {d[key]}")

student_id  : S2026001
first_name  : Sara
last_name   : Ahmadi
major       : Computer Science
year        : 2
address     : {'street': '12 Valiasr St', 'city': 'Tehran', 'postal_code': '1599616313', 'country': 'Iran'}
enrollments : [{'course_code': 'CS101', 'semester': '2025-Fall', 'grade': 'A'}, {'course_code': 'MATH201', 'semester': '2025-Fall', 'grade': 'B+'}, {'course_code': 'ENG100', 'semester': '2025-Fall', 'grade': 'A-'}]


Notice two choices:
- We don't save `gpa` — it's **computed**, so it will be recomputed after loading.
- For a course we save only its **code**. When loading, we look it up in the catalogue. That way all students share the same course objects again.

### Loading: from dictionary back to objects

In [48]:
def student_from_dict(d, catalogue):
    address = None
    if d["address"] is not None:
        a = d["address"]
        address = Address(a["street"], a["city"], a["postal_code"], a["country"])

    student = Student(d["student_id"], d["first_name"], d["last_name"],
                      d["major"], d["year"], address)        # validation runs again!

    for e in d["enrollments"]:
        course = catalogue[e["course_code"]]
        enrollment = student.enroll(course, e["semester"])
        enrollment.grade = e["grade"]
    return student

copy_of_sara = student_from_dict(sara.to_dict(), CATALOGUE)
print(copy_of_sara.transcript())
print("\nEqual to the original?", copy_of_sara == sara, "| same object?", copy_of_sara is sara)

Sara Ahmadi (S2026001) - Computer Science, year 2
   address: 12 Valiasr St, Tehran 1599616313, Iran
   2025-Fall: CS101 Intro to Programming with Python (4 cr) -> A
   2025-Fall: MATH201 Linear Algebra (3 cr) -> B+
   2025-Fall: ENG100 Academic Writing (2 cr) -> A-
   GPA 3.70 | credits earned 9

Equal to the original? True | same object? False


### Writing to and reading from a JSON file

In [49]:
import json

students_to_save = [sara, ali]

# --- save ---
data = []
for s in students_to_save:
    data.append(s.to_dict())
with open("students.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2)
print("Saved", len(data), "students to students.json")

# --- look at the file ---
with open("students.json", encoding="utf-8") as file:
    text = file.read()
print(text[:400], "...")

Saved 2 students to students.json
[
  {
    "student_id": "S2026001",
    "first_name": "Sara",
    "last_name": "Ahmadi",
    "major": "Computer Science",
    "year": 2,
    "address": {
      "street": "12 Valiasr St",
      "city": "Tehran",
      "postal_code": "1599616313",
      "country": "Iran"
    },
    "enrollments": [
      {
        "course_code": "CS101",
        "semester": "2025-Fall",
        "grade": "A"
      }, ...


In [50]:
# --- load ---
with open("students.json", encoding="utf-8") as file:
    loaded_data = json.load(file)

loaded_students = []
for d in loaded_data:
    loaded_students.append(student_from_dict(d, CATALOGUE))

for s in loaded_students:
    print(s)

Sara Ahmadi (S2026001), Computer Science, GPA 3.7
Ali Rezaei (S2026002), Mathematics, GPA 0.0


---
## Part 15 — Mini-project: a `StudentRegistry`

We now build the registrar's office: a class that **manages many students**. It stores them in a dictionary `{student_id: Student}` so that finding a student by ID is fast.

| Operation | Method |
|---|---|
| **Insert** | `add(student)` — refuses duplicate IDs |
| **Access** | `get(student_id)`, `count()` |
| **Search** | `search_by_name(text)`, `students_in_major(major)`, `students_with_gpa_at_least(value)` |
| **Update** | `change_major(student_id, new_major)`, `record_grade(...)` |
| **Delete** | `remove(student_id)` |
| **Reports** | `ranking()`, `honor_roll()`, `average_gpa_by_major()`, `course_roster(code)` |
| **Save / load** | `save(filename)`, `load(filename)` |

In [51]:
class StudentRegistry:
    def __init__(self, university_name, catalogue):
        self.university_name = university_name
        self.catalogue = catalogue
        self.students = {}                     # student_id -> Student

    # ---------------- insert ----------------
    def add(self, student):
        if student.student_id in self.students:
            raise ValueError(f"a student with ID {student.student_id} already exists")
        self.students[student.student_id] = student

    # ---------------- access ----------------
    def get(self, student_id):
        if student_id not in self.students:
            raise LookupError(f"no student with ID {student_id}")
        return self.students[student_id]

    def count(self):
        return len(self.students)

    def all_students(self):
        return list(self.students.values())

    # ---------------- search ----------------
    def search_by_name(self, text):
        result = []
        for s in self.students.values():
            if text.lower() in s.full_name().lower():
                result.append(s)
        return result

    def students_in_major(self, major):
        result = []
        for s in self.students.values():
            if s.major == major:
                result.append(s)
        return result

    def students_with_gpa_at_least(self, value):
        result = []
        for s in self.students.values():
            if s.gpa() >= value:
                result.append(s)
        return result

    # ---------------- update ----------------
    def change_major(self, student_id, new_major):
        student = self.get(student_id)
        student.major = new_major
        return student

    def record_grade(self, student_id, course_code, semester, grade):
        self.get(student_id).set_grade(course_code, semester, grade)

    # ---------------- delete ----------------
    def remove(self, student_id):
        student = self.get(student_id)            # raises LookupError if missing
        del self.students[student_id]
        return student

    # ---------------- reports ----------------
    def ranking(self):
        def gpa_key(student):
            return student.gpa()
        return sorted(self.students.values(), key=gpa_key, reverse=True)

    def honor_roll(self, threshold=3.5):
        result = []
        for s in self.ranking():
            if s.gpa() >= threshold:
                result.append(s)
        return result

    def average_gpa_by_major(self):
        totals = {}                                # major -> [sum of GPAs, number of students]
        for s in self.students.values():
            if s.major not in totals:
                totals[s.major] = [0.0, 0]
            totals[s.major][0] = totals[s.major][0] + s.gpa()
            totals[s.major][1] = totals[s.major][1] + 1
        averages = {}
        for major in sorted(totals):
            gpa_sum, how_many = totals[major]
            averages[major] = round(gpa_sum / how_many, 2)
        return averages

    def course_roster(self, course_code):
        roster = []
        for s in self.students.values():
            e = s.find_enrollment(course_code)
            if e is not None:
                roster.append((s.full_name(), e.grade))
        return sorted(roster)

    def print_table(self):
        print(f"{'ID':<10}{'Name':<16}{'Major':<18}{'Year':>4}{'GPA':>6}")
        print("-" * 54)
        for s in self.ranking():
            print(f"{s.student_id:<10}{s.full_name():<16}{s.major:<18}{s.year:>4}{s.gpa():>6.2f}")

    # ---------------- save / load ----------------
    def save(self, filename):
        data = {"university": self.university_name, "students": []}
        for s in self.students.values():
            data["students"].append(s.to_dict())
        with open(filename, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2)

    def load(self, filename):
        with open(filename, encoding="utf-8") as file:
            data = json.load(file)
        self.university_name = data["university"]
        self.students = {}
        for d in data["students"]:
            self.add(student_from_dict(d, self.catalogue))

    def __str__(self):
        return f"StudentRegistry of {self.university_name} with {self.count()} students"

print("StudentRegistry ready")

StudentRegistry ready


### Fill the registry with sample data

In [52]:
registry = StudentRegistry("Example University", CATALOGUE)

sample = [
    ("S2026001", "Sara",  "Ahmadi",  "Computer Science", 2, [("CS101", "A"), ("MATH201", "B+"), ("ENG100", "A-")]),
    ("S2026002", "Ali",   "Rezaei",  "Mathematics",      1, [("MATH201", "A"), ("STAT210", "A-"), ("CS101", "B")]),
    ("S2026003", "Emma",  "Wilson",  "Physics",          3, [("PHYS110", "B"), ("MATH201", "C+"), ("ENG100", "B-")]),
    ("S2026004", "Liam",  "Chen",    "Computer Science", 2, [("CS101", "A"), ("CS201", "A"), ("STAT210", "A")]),
    ("S2026005", "Neda",  "Karimi",  "Mathematics",      4, [("MATH201", "B"), ("STAT210", "C"), ("PHYS110", "D")]),
    ("S2026006", "Omid",  "Tehrani", "Computer Science", 1, [("CS101", "B+"), ("ENG100", "A")]),
]
for student_id, first, last, major, year, grades in sample:
    s = Student(student_id, first, last, major, year)
    for course_code, letter in grades:
        s.enroll(CATALOGUE[course_code], "2025-Fall")
        s.set_grade(course_code, "2025-Fall", letter)
    registry.add(s)

print(registry)
print()
registry.print_table()

StudentRegistry of Example University with 6 students

ID        Name            Major             Year   GPA
------------------------------------------------------
S2026004  Liam Chen       Computer Science     2  4.00
S2026001  Sara Ahmadi     Computer Science     2  3.70
S2026006  Omid Tehrani    Computer Science     1  3.53
S2026002  Ali Rezaei      Mathematics          1  3.51
S2026003  Emma Wilson     Physics              3  2.70
S2026005  Neda Karimi     Mathematics          4  1.90


### Operation tour

In [53]:
print("▶ Access")
print("  get('S2026004')       :", registry.get("S2026004"))
try:
    registry.get("S9999999")
except LookupError as error:
    print("  get('S9999999')       : LookupError:", error)

print("\n▶ Search")
names = []
for s in registry.search_by_name("ra"):
    names.append(s.full_name())
print("  names containing 'ra' :", names)
print("  Mathematics majors    :", [s.full_name() for s in registry.students_in_major("Mathematics")])
print("  GPA at least 3.5      :", [s.full_name() for s in registry.students_with_gpa_at_least(3.5)])

▶ Access
  get('S2026004')       : Liam Chen (S2026004), Computer Science, GPA 4.0
  get('S9999999')       : LookupError: no student with ID S9999999

▶ Search
  names containing 'ra' : ['Sara Ahmadi', 'Omid Tehrani']
  Mathematics majors    : ['Ali Rezaei', 'Neda Karimi']
  GPA at least 3.5      : ['Sara Ahmadi', 'Ali Rezaei', 'Liam Chen', 'Omid Tehrani']


> 💡 The last two lines use a **list comprehension**: `[s.full_name() for s in some_list]` is a short way to write a loop that builds a list. It does the same as the `names = []` / `for` / `append` lines above it.

In [54]:
print("▶ Insert: a duplicate ID is refused")
try:
    registry.add(Student("S2026001", "Fake", "Sara", "History"))
except ValueError as error:
    print("  ValueError:", error)

print("\n▶ Update: Ali switches to Statistics; Emma takes a new course and gets a grade")
registry.change_major("S2026002", "Statistics")
emma = registry.get("S2026003")
emma.enroll(CATALOGUE["STAT210"], "2026-Spring")
registry.record_grade("S2026003", "STAT210", "2026-Spring", "A")
print("  ", registry.get("S2026002"))
print("  ", emma)

print("\n▶ Delete: Omid withdraws")
removed = registry.remove("S2026006")
print("   removed", removed.full_name(), "->", registry)

print("\n▶ Reports")
print("   Honor roll       :", [(s.full_name(), s.gpa()) for s in registry.honor_roll()])
print("   Avg GPA by major :", registry.average_gpa_by_major())
print("   CS101 roster     :", registry.course_roster("CS101"))
print()
registry.print_table()

▶ Insert: a duplicate ID is refused
  ValueError: a student with ID S2026001 already exists

▶ Update: Ali switches to Statistics; Emma takes a new course and gets a grade
   Ali Rezaei (S2026002), Statistics, GPA 3.51
   Emma Wilson (S2026003), Physics, GPA 3.02

▶ Delete: Omid withdraws
   removed Omid Tehrani -> StudentRegistry of Example University with 5 students

▶ Reports
   Honor roll       : [('Liam Chen', 4.0), ('Sara Ahmadi', 3.7), ('Ali Rezaei', 3.51)]
   Avg GPA by major : {'Computer Science': 3.85, 'Mathematics': 1.9, 'Physics': 3.02, 'Statistics': 3.51}
   CS101 roster     : [('Ali Rezaei', 'B'), ('Liam Chen', 'A'), ('Sara Ahmadi', 'A')]

ID        Name            Major             Year   GPA
------------------------------------------------------
S2026004  Liam Chen       Computer Science     2  4.00
S2026001  Sara Ahmadi     Computer Science     2  3.70
S2026002  Ali Rezaei      Statistics           1  3.51
S2026003  Emma Wilson     Physics              3  3.02
S2026005

In [55]:
print("▶ Save and load")
registry.save("registry.json")

new_registry = StudentRegistry("(empty)", CATALOGUE)
new_registry.load("registry.json")
print("   loaded:", new_registry)

all_equal = True
for s in registry.all_students():
    loaded = new_registry.get(s.student_id)
    if loaded.gpa() != s.gpa() or loaded.major != s.major:
        all_equal = False
print("   every student restored with the same data:", all_equal)

▶ Save and load
   loaded: StudentRegistry of Example University with 5 students
   every student restored with the same data: True


---
## Part 16 — Summary

### What a hand-written data class usually contains

```python
class Student:
    UNIVERSITY = "Example University"             # class attribute (shared)

    def __init__(self, student_id, name, courses=None):
        self.student_id = check_id(student_id)     # validate
        self.name = name                           # store data
        self.courses = [] if courses is None else courses   # safe mutable default

    def gpa(self): ...                             # computed value
    def add_course(self, code): ...                # actions that change the data
    def __str__(self): ...                         # print(student)
    def __repr__(self): ...                        # how it looks to programmers
    def __eq__(self, other): ...                   # student1 == student2
    def __hash__(self): ...                        # use in sets / as dict keys
    def __lt__(self, other): ...                   # sorting
    def to_dict(self): ...                         # saving
```

### Special ("dunder") methods we used

| Method | Called when you write… | Must return |
|---|---|---|
| `__init__(self, ...)` | `Student(...)` | nothing |
| `__str__(self)` | `print(s)`, `str(s)`, `f"{s}"` | a string |
| `__repr__(self)` | `repr(s)`, showing `s` in a list or as the last line of a cell | a string |
| `__eq__(self, other)` | `s1 == s2`, `s1 != s2`, `s in list` | `True` / `False` |
| `__hash__(self)` | `{s1, s2}`, `d[s] = ...` | an integer (use `hash(...)`) |
| `__lt__(self, other)` | `s1 < s2`, `sorted(list)`, `min`, `max` | `True` / `False` |

### Common mistakes

| Mistake | What happens | Fix |
|---|---|---|
| forgetting `self` in a method definition | `TypeError: ... takes 0 positional arguments but 1 was given` | first parameter is always `self` |
| writing `name = name` instead of `self.name = name` | attribute is never stored | always `self.` |
| `courses=[]` as a default | all students share one list | `courses=None`, then create `[]` inside |
| `__str__` that **prints** instead of **returns** | `TypeError: __str__ returned non-string` | `return` a string |
| defining `__eq__` but not `__hash__` | `TypeError: unhashable type` in sets/dicts | add `__hash__` on the same (unchanging) attributes |
| `self.counter += 1` for a shared counter | creates a private copy on one object | `ClassName.counter += 1` |
| forgetting `super().__init__(...)` in a child | parent attributes missing | always call it first |
| storing a value that can be computed (GPA) | the value gets out of date | compute it in a method |
| changing data directly, skipping validation | bad data sneaks in | change data through methods that check it |

> 🔭 **Looking ahead.** Once you are comfortable writing these methods by hand, you will discover that Python can generate `__init__`, `__repr__` and `__eq__` automatically for simple data classes. That shortcut does exactly what you wrote yourself in this notebook — so you will understand it right away.

### Exercises

1. **Course validation.** Make `Course.__init__` refuse a code that is not 2–4 capital letters followed by 3 digits (`CS101` ✓, `MATH201` ✓, `cs101` ✗, `COMPSCI101` ✗).
2. **Semester GPA.** Add a method `semester_gpa(semester)` to `Student` that only counts enrollments of that semester.
3. **Prerequisites.** Give `Course` an extra parameter `prerequisites=None` (a list of course codes). Make a new `enroll` that refuses a course if the student has not passed every prerequisite with a grade other than `F`.
4. **Professor's teaching load.** Write a `Professor(Person)` class with a list of course codes it teaches (safe default!), a method `add_course(code)`, and a method `teaching_credits()` that adds up the credits using `CATALOGUE`.
5. **Natural order.** Add `__lt__` to `Course` so that `sorted(CATALOGUE.values())` orders courses by department, then by code.
6. **Report card.** Add a method `report_card()` to `StudentRegistry` that returns a text with one line per student: rank, name, GPA and standing (`Dean's List` / `Good standing` / `Academic probation`).

### Solutions
Try the exercises yourself first! Then compare with the solutions below.

In [56]:
# ---------- Exercise 1: Course validation ----------
def is_valid_course_code(code):
    letters = ""
    i = 0
    while i < len(code) and code[i].isalpha():
        letters = letters + code[i]
        i = i + 1
    digits = code[i:]
    return (2 <= len(letters) <= 4 and letters.isupper()
            and len(digits) == 3 and digits.isdigit())

class CheckedCourse(Course):
    def __init__(self, code, title, credits, department, prerequisites=None):
        if not is_valid_course_code(code):
            raise ValueError(f"bad course code {code!r}")
        super().__init__(code, title, credits, department)
        if prerequisites is None:
            prerequisites = []
        self.prerequisites = prerequisites           # used in Exercise 3

for code in ["CS101", "MATH201", "cs101", "COMPSCI101", "CS10"]:
    try:
        CheckedCourse(code, "Test", 3, "Test")
        print("Exercise 1:", code, "accepted")
    except ValueError as error:
        print("Exercise 1:", error)

Exercise 1: CS101 accepted
Exercise 1: MATH201 accepted
Exercise 1: bad course code 'cs101'
Exercise 1: bad course code 'COMPSCI101'
Exercise 1: bad course code 'CS10'


In [57]:
# ---------- Exercises 2 and 3: semester GPA and prerequisites ----------
class Student2(Student):
    def semester_gpa(self, semester):
        total_credits = 0
        weighted_points = 0
        for e in self.enrollments:
            if e.semester == semester and e.is_completed():
                total_credits = total_credits + e.course.credits
                weighted_points = weighted_points + GRADE_POINTS[e.grade] * e.course.credits
        if total_credits == 0:
            return 0.0
        return round(weighted_points / total_credits, 2)

    def has_passed(self, course_code):
        for e in self.enrollments:
            if e.course.code == course_code and e.is_completed() and e.grade != "F":
                return True
        return False

    def enroll(self, course, semester):
        prerequisites = getattr(course, "prerequisites", [])   # older Course objects have none
        for code in prerequisites:
            if not self.has_passed(code):
                raise ValueError(f"cannot enroll in {course.code}: prerequisite {code} not passed")
        return super().enroll(course, semester)

intro = CheckedCourse("CS101", "Intro to Programming", 4, "Computer Science")
data_structures = CheckedCourse("CS201", "Data Structures", 4, "Computer Science", ["CS101"])

test = Student2("S2026010", "Test", "Student", "Computer Science")
try:
    test.enroll(data_structures, "2025-Fall")
except ValueError as error:
    print("Exercise 3:", error)

test.enroll(intro, "2025-Fall")
test.set_grade("CS101", "2025-Fall", "A")
test.enroll(data_structures, "2026-Spring")
test.set_grade("CS201", "2026-Spring", "B")
print("Exercise 3: now enrolled in", [e.course.code for e in test.enrollments])
print("Exercise 2: 2025-Fall GPA =", test.semester_gpa("2025-Fall"),
      "| 2026-Spring GPA =", test.semester_gpa("2026-Spring"), "| overall =", test.gpa())

Exercise 3: cannot enroll in CS201: prerequisite CS101 not passed
Exercise 3: now enrolled in ['CS101', 'CS201']
Exercise 2: 2025-Fall GPA = 4.0 | 2026-Spring GPA = 3.0 | overall = 3.5


In [58]:
# ---------- Exercise 4: Professor with a teaching load ----------
class TeachingProfessor(Person):
    def __init__(self, name, email, staff_id, courses=None):
        super().__init__(name, email)
        self.staff_id = staff_id
        if courses is None:
            courses = []
        self.courses = courses

    def add_course(self, code):
        if code not in CATALOGUE:
            raise ValueError(f"unknown course {code}")
        if code not in self.courses:
            self.courses.append(code)

    def teaching_credits(self):
        total = 0
        for code in self.courses:
            total = total + CATALOGUE[code].credits
        return total

prof = TeachingProfessor("Liam Chen", "l.chen@uni.example.edu", "P0042")
prof.add_course("CS101")
prof.add_course("CS201")
prof.add_course("CS101")          # duplicate ignored
print("Exercise 4:", prof, "| teaches", prof.courses, "| credits:", prof.teaching_credits())

# ---------- Exercise 5: natural order of courses ----------
def course_less_than(self, other):
    return (self.department, self.code) < (other.department, other.code)

Course.__lt__ = course_less_than
print("Exercise 5:")
for c in sorted(CATALOGUE.values()):
    print("   ", c.department.ljust(17), c)

Exercise 4: Liam Chen <l.chen@uni.example.edu> | teaches ['CS101', 'CS201'] | credits: 8
Exercise 5:
    Computer Science  CS101 Intro to Programming with Python (4 cr)
    Computer Science  CS201 Data Structures (4 cr)
    Humanities        ENG100 Academic Writing (2 cr)
    Mathematics       MATH201 Linear Algebra (3 cr)
    Mathematics       STAT210 Probability and Statistics (3 cr)
    Physics           PHYS110 Physics I (4 cr)


In [59]:
# ---------- Exercise 6: report card ----------
def standing_for(gpa):
    if gpa >= 3.5:
        return "Dean's List"
    if gpa >= 2.0:
        return "Good standing"
    return "Academic probation"

def report_card(self):
    lines = [f"Report card - {self.university_name}"]
    rank = 1
    for s in self.ranking():
        lines.append(f"{rank:>2}. {s.full_name():<16} GPA {s.gpa():.2f}  {standing_for(s.gpa())}")
        rank = rank + 1
    return "\n".join(lines)

StudentRegistry.report_card = report_card
print(registry.report_card())

Report card - Example University
 1. Liam Chen        GPA 4.00  Dean's List
 2. Sara Ahmadi      GPA 3.70  Dean's List
 3. Ali Rezaei       GPA 3.51  Dean's List
 4. Emma Wilson      GPA 3.02  Good standing
 5. Neda Karimi      GPA 1.90  Academic probation


---
### 🎓 Congratulations!

You have built, completely by hand, a set of data classes for a university:

- `Course`, `Address`, `Enrollment` and `Student`, with **validation**, **computed values**, **nice printing** and **comparison**;
- a class family `Person → Student → GraduateStudent` using **inheritance**;
- a `StudentRegistry` that can **add, find, update, delete, report, save and load** students.

These ideas (constructor, methods, special methods, composition, inheritance) are the foundation of object-oriented programming in Python and in almost every other language.